# Notebook19 — computation optimization and global-prior pilot

Run All launches the authorized workflow in detached tmux. It verifies numerical correctness and exact checkpoint recovery, profiles complete JEPA updates, then compares ViT, the optimized learned kernel with an initial 50% local weight, and the same kernel with a 10% local weight. Rank 64 and all other options stay fixed. Both learned encoders add 98,328 online parameters.

Two fresh paired seeds use 256 px images, 8 px patches, FP32, physical batch 16 with accumulation 8, and the same four-worker input pipeline. We evaluate untrained controls, a pinned 200-update checkpoint and the final 900-second active-wall checkpoint with TRAIN-fitted linear and nearest-neighbour readouts on DEV. This pilot provides descriptive evidence; it makes no statistical significance claim. Official test pixels remain closed.

Expected experiment runtime: 2–3 hours, with a hard original three-hour cap after the short discarded checkpoint proof. Startup may queue while GPU 1 is actively used; queue waiting is recorded separately. GPU 0 remains assigned to the other AHR job. Compute profiling and real data preparation costs are reported separately. Continuous 1 Hz GPU observation qualifies speed claims conditionally; it cannot rule out subsecond interference.

Artifacts: `artifacts/notebook_19/<RUN_ID>/report.html` and source-bound records. Checkpoints: `checkpoints/<RUN_ID>/`, retaining latest two, best and the pinned 200-update state. Save every 50 updates, epoch boundary and at least every 15 minutes. Exact resume: keep the same RUN_ID, sources, options and original clock, restart the `ahr-go` kernel and Run All. Invalid or IN_FLIGHT timing never silently restarts.

Detached sessions: `fmm19-compute-gate-002`, `-guard`, `-worker0` and `-launcher`. Attach with `tmux attach -t fmm19-compute-gate-002`; detach with Ctrl-b then d. The workspace is on the persistent boot disk; copy final/best checkpoints off-instance and verify hashes before recycling.


In [ ]:
from pathlib import Path
import os,json,types,nbformat
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
RUN_ID=os.environ.get('FMM19_RUN_ID','compute-gate-002')
PROFILE=os.environ.get('FMM19_PROFILE','full')
P=ROOT/'experiments/19_compute_and_gate/workflow.ipynb'
W=types.ModuleType('fmm19_frontdoor');W.__file__=str(P)
for cell in nbformat.read(P,4).cells:
    if cell.cell_type=='code'and'ahr-export'in cell.metadata.get('tags',[]):exec(compile(cell.source,str(P),'exec'),W.__dict__)
print({'run_id':RUN_ID,'profile':PROFILE,'pilot_maximum_hours':3,'paired_seeds':2,'conditions':3,'active_seconds_per_model':900,'matched_updates':200})


### Launch or exact resume

In [ ]:
LAUNCH=W.launch_study(ROOT,RUN_ID,PROFILE)
print(json.dumps(LAUNCH,indent=2,default=str))

### Current committed status

In [ ]:
ARTIFACT=ROOT/'artifacts/notebook_19'/RUN_ID
for name in ('study_status.json','guard_heartbeat.json','launcher_heartbeat.json'):
    path=ARTIFACT/name
    if path.exists():print(name,json.dumps(json.loads(path.read_text()),indent=2))
print('Visual report:',ARTIFACT/'report.html')
